In [ ]:
!nvidia-smi


Tue Oct  6 19:38:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!python --version

Python 3.13.15


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone -b ayan https://github.com/AyanAlam99/llm-emotion-representations.git /content/llm-emotion-representations

fatal: destination path '/content/llm-emotion-representations' already exists and is not an empty directory.


In [ ]:
%cd /content/llm-emotion-representations

/content/llm-emotion-representations


In [ ]:
!pip install -q -U "transformers>=5.18.0" accelerate
!pip install -q -e . --no-deps --ignore-requires-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 117.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llm-emotion-representations (pyproject.toml) ... done


In [ ]:
!sed -i '/"presence_penalty"/d' src/emotion_vectors/config.py
!sed -i 's/chat_template_kwargs={"enable_thinking": thinking_enabled}/enable_thinking=thinking_enabled/' src/emotion_vectors/generation.py

In [ ]:
!grep -n "presence_penalty\|enable_thinking" src/emotion_vectors/*.py

src/emotion_vectors/generation.py:46:        enable_thinking=thinking_enabled,


In [ ]:
RUN = "/content/drive/MyDrive/emotion_runs/pilot"

In [ ]:
!python scripts/generate_dataset.py --run-dir {RUN} --limit 11 --torch-dtype bfloat16

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 320/320 [00:03<00:00, 95.31it/s] 
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-l

In [ ]:
!python scripts/extract_activations.py --run-dir {RUN} --limit 11 --torch-dtype bfloat16

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 320/320 [00:01<00:00, 272.39it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
extracted 1/11: neutral_a_pet_A_0042 shape=(24, 2048)
extracted 2/11: anger_a_pet_A_0042 shape=(24, 2048)
extracted 3/11: annoyance_a_pet_A_0042 shape=(24, 2048)
extracted 4/11: fear_a_pet_A_0042 shape=(24, 2048)
extracted 5/11: nervousness_a_pet_A_0042 shape=(24, 2048)
extracted 6/11: sadness_a_pet_A_0042 shape=(24, 2048)
extracted 7/11: disappointment_a_pet_A_0042 shape=(24, 2048)
extracted 8/11: joy_a_pet_A_0042 shape=

In [ ]:
# Cell 7: pilot output check
import json, torch, sys
sys.path.insert(0, "src")
from emotion_vectors.vectors import build_emotion_vectors

recs = [json.loads(l) for l in open(f"{RUN}/generations.jsonl")]
for r in recs[:3]:
    print(r["sample_id"], r["validation"])
    print(r["response"][:], "\n---")
print(repr(recs[0]["rendered_prompt"][-60:]))

acts = {r["sample_id"]: torch.load(f"{RUN}/activations/{r['sample_id']}.pt")
        for r in recs if r["validation"]["valid"]}
print(next(iter(acts.values())).shape)
raw, unit, pair_ids = build_emotion_vectors(acts, recs)
print({e: tuple(v.shape) for e, v in unit.items()})

neutral_a_pet_A_0042 {'valid': True, 'errors': [], 'warnings': []}
The morning light filtered through the window, illuminating the wooden floorboards. I walked to the door and opened it to let the house warm up. There was a fresh bag of treats waiting on the counter. I sat on my cushion, waiting for the mailman. He arrived just as the sun began to rise. He left with a slight smile as he closed the front door behind him. I turned off the lamp and closed my eyes. The day had begun. It was simply another day in the quiet life of a small owner and their pet. The routine continued without any sudden changes or unexpected events. 
---
anger_a_pet_A_0042 {'valid': True, 'errors': [], 'warnings': []}
The leash slipped from my fingers, and the dog's massive head whipped toward me, eyes wild with frustration. I stepped back, my knuckles white, but the animal was not waiting for my patience to recover. One moment we were calm, the next a surge of pure rage burned through my veins, hot and sudden.